# Lecture 4 · Notebook 1 — Self-attention from scratch
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NLP-Course-CUHKSZ/NLP-course-cuhksz.github.io/blob/main/materials/tutorials/lecture04/lecture04_1_self_attention_from_scratch.ipynb)

**Companion to [Chapter 8 of the video lecture](https://nlp-course-cuhksz.github.io/lecture04.html#p8-s91)** (slides 90–105), with one step back to the decoder mask of
[Chapter 7](https://nlp-course-cuhksz.github.io/lecture04.html#p7-s86). Self-attention is the only place in a Transformer block where tokens exchange information
(*composition*); the feed-forward network then transforms each token on its own (*abstraction*). In this notebook you will:

1. reproduce the numbers on slides 92–94 (scores 112 and 96 → weights 0.88 and 0.12),
2. see **why** the scores are divided by √d<sub>k</sub>,
3. write a self-attention layer in a few lines of PyTorch and check it against PyTorch's built-in kernel,
4. discover that attention alone **ignores word order**, and add a **causal mask** like a GPT decoder,
5. build **multi-head attention** and match `torch.nn.MultiheadAttention` exactly, and
6. train a one-layer attention model on a long-range task — with and without position information.

Everything runs on CPU in about a minute. No downloads.

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
print("torch", torch.__version__)

## 1 · The worked example on slides 92–94
For the input *Thinking Machines*, the query of *Thinking* has dot product **112** with its own key and **96** with the key of
*Machines*. With d<sub>k</sub> = 64 we divide by √64 = 8, then apply a softmax, then mix the value vectors.

In [ ]:
scores = torch.tensor([112., 96.])
d_k = 64
scaled = scores / math.sqrt(d_k)
weights = scaled.softmax(-1)
print("scaled scores:", scaled.tolist())            # [14, 12]
print("attention    :", [round(w, 2) for w in weights.tolist()])   # [0.88, 0.12]

# z1 = 0.88 * v1 + 0.12 * v2  (two made-up 3-d value vectors, as on slide 94)
v = torch.tensor([[1., 0., 2.], [0., 3., 1.]])
z1 = weights @ v
print("z1           :", z1.tolist())

A score gap of only 2 became 0.88 vs. 0.12: the softmax keeps the ranking and **exaggerates** gaps, because it exponentiates.

## 2 · Why divide by √d<sub>k</sub>?
If the components of q and k are independent with mean 0 and variance 1, the dot product q·k has variance d<sub>k</sub>. Large
scores push the softmax towards a one-hot vector, whose gradients are tiny (slide 93). Let us measure it.

In [ ]:
n_keys = 10
print(f"{'d_k':>5} | {'std(q.k)':>8} | max weight (unscaled) | max weight (scaled)")
for d in [4, 16, 64, 256, 1024]:
    q, K = torch.randn(2000, d), torch.randn(2000, n_keys, d)
    s = torch.einsum("bd,bnd->bn", q, K)
    print(f"{d:5d} | {s.std():8.1f} | {s.softmax(-1).max(-1).values.mean():21.2f} | {(s / math.sqrt(d)).softmax(-1).max(-1).values.mean():.2f}")

Without scaling, the largest weight approaches 1 as d<sub>k</sub> grows (attention becomes a hard choice). With scaling, the
distribution stays equally soft for every width.

## 3 · A self-attention layer in eight lines
Each token vector x is projected three times (slide 91): a **query** (what am I looking for?), a **key** (what do I offer?) and a
**value** (what do I hand over?). Then `softmax(Q Kᵀ / √d_k) V`.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super().__init__()
        self.W_q, self.W_k, self.W_v = (nn.Linear(d_model, d_k, bias=False) for _ in range(3))
    def forward(self, X, mask=None):                  # X: [n, d_model]
        Q, K, V = self.W_q(X), self.W_k(X), self.W_v(X)
        S = Q @ K.T / math.sqrt(K.shape[-1])          # [n, n] scores: one row per query
        if mask is not None:
            S = S.masked_fill(~mask, float("-inf"))   # blocked pairs get exactly zero weight
        A = S.softmax(dim=-1)                         # each row is a probability distribution
        return A @ V, A

words = "Nobel committee awards Strickland who advanced optics".split()
n, d_model, d_k = len(words), 16, 8
X = torch.randn(n, d_model)                           # stand-in word embeddings (untrained)
attn = SelfAttention(d_model, d_k)
Z, A = attn(X)
print("output:", tuple(Z.shape), " row sums:", A.sum(-1).detach().round(decimals=4).tolist())

ref = F.scaled_dot_product_attention(attn.W_q(X)[None], attn.W_k(X)[None], attn.W_v(X)[None])[0]
print("matches torch's scaled_dot_product_attention:", torch.allclose(Z, ref, atol=1e-6))

plt.figure(figsize=(4.5, 4)); plt.imshow(A.detach(), cmap="Purples")
plt.xticks(range(n), words, rotation=45, ha="right"); plt.yticks(range(n), words)
plt.xlabel("key (attended to)"); plt.ylabel("query"); plt.title("random, untrained weights"); plt.colorbar(); plt.show()

The weights are random, so the pattern means nothing yet — **training** decides which token *who* looks at (Notebook 3 inspects a
trained model). Note that all seven rows were computed at once with two matrix products: no step depends on the previous token,
unlike an RNN (slide 98).

## 4 · Attention alone ignores word order
Shuffle the input tokens. If self-attention had any notion of order, the outputs would change in some complex way. Instead they are
simply shuffled the same way (*permutation equivariance*). This is why the Transformer **adds position embeddings** (slide 81).

In [ ]:
perm = torch.randperm(n)
Z_perm, _ = attn(X[perm])
print("shuffled input  ->  same outputs, shuffled:", torch.allclose(Z_perm, Z[perm], atol=1e-6))

def sinusoidal(n_pos, d):                             # the original encoding (slide 106); Notebook 2 explores it
    pos = torch.arange(n_pos)[:, None]
    rate = 10000 ** (-torch.arange(0, d, 2) / d)
    pe = torch.zeros(n_pos, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * rate), torch.cos(pos * rate)
    return pe

P = sinusoidal(n, d_model)
Z_pos, _ = attn(X + P)
Z_pos_perm, _ = attn(X[perm] + P)                    # same words, new order, positions added
print("with position embeddings, still just shuffled? ", torch.allclose(Z_pos_perm, Z_pos[perm], atol=1e-6))

## 5 · The causal mask of a decoder (slide 86)
A GPT-style decoder may only look at itself and earlier positions. We set the blocked scores to −∞ *before* the softmax, so they get
exactly zero weight. Then changing a **later** token cannot change any **earlier** output — which keeps parallel training honest.

In [ ]:
causal = torch.tril(torch.ones(n, n, dtype=torch.bool))
Zc, Ac = attn(X, mask=causal)

X2 = X.clone(); X2[-1] = torch.randn(d_model)        # change only the last word
Zc2, _ = attn(X2, mask=causal)
print("outputs 0..5 unchanged:", torch.allclose(Zc[:-1], Zc2[:-1]), "| output 6 changed:", not torch.allclose(Zc[-1], Zc2[-1]))

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, (M, t) in zip(ax, [(A, "encoder: no mask"), (Ac, "decoder: causal mask")]):
    a.imshow(M.detach(), cmap="Purples"); a.set_title(t)
    a.set_xticks(range(n)); a.set_xticklabels(words, rotation=45, ha="right"); a.set_yticks(range(n)); a.set_yticklabels(words)
plt.tight_layout(); plt.show()

## 6 · Multi-head attention (slides 103–104)
Run H attention functions in parallel, each with its own (smaller) Q, K, V projections; **concatenate** the H outputs and apply a
learned output projection W<sub>O</sub>. With d<sub>k</sub> = d<sub>model</sub> / H, the cost is similar to one big head.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model)  # all heads' Q, K, V in one matrix
        self.W_o = nn.Linear(d_model, d_model)
    def forward(self, X, mask=None):                  # X: [n, d_model] or [batch, n, d_model]
        *b, n, _ = X.shape
        Q, K, V = self.W_qkv(X).chunk(3, dim=-1)
        Q, K, V = (t.reshape(*b, n, self.h, self.d_k).transpose(-3, -2) for t in (Q, K, V))   # [..., h, n, d_k]
        S = Q @ K.transpose(-1, -2) / math.sqrt(self.d_k)                                     # [..., h, n, n]
        if mask is not None:
            S = S.masked_fill(~mask, float("-inf"))
        A = S.softmax(-1)
        heads = A @ V                                                                         # [..., h, n, d_k]
        concat = heads.transpose(-3, -2).reshape(*b, n, self.h * self.d_k)                    # concatenate, don't average
        return self.W_o(concat), A

mha = MultiHeadAttention(d_model=16, n_heads=4)
out, A_heads = mha(X)
print("output", tuple(out.shape), "| attention maps", tuple(A_heads.shape))

# Same weights inside PyTorch's implementation -> identical output
ref = nn.MultiheadAttention(16, 4, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(mha.W_qkv.weight); ref.in_proj_bias.copy_(mha.W_qkv.bias)
    ref.out_proj.weight.copy_(mha.W_o.weight); ref.out_proj.bias.copy_(mha.W_o.bias)
ref_out, _ = ref(X[None], X[None], X[None])
print("matches nn.MultiheadAttention:", torch.allclose(out, ref_out[0], atol=1e-6))

one_head = MultiHeadAttention(16, 1)
print("parameters, 4 heads:", sum(p.numel() for p in mha.parameters()), "| 1 head:", sum(p.numel() for p in one_head.parameters()))

Four heads cost exactly as many parameters as one head of full width — they just split the dimensions differently, so that each
head can produce its own attention distribution.

## 7 · Train it: a long-range task
Task: read a random sequence of 16 tokens and, at the **last** position, output the **first** token. A CNN with a small window or an RNN
would have to carry that token across 15 steps (Chapter 2, slide 27); one attention layer can fetch it directly in one step. We train
the same tiny model twice — with and without learned position embeddings.

In [ ]:
V_SIZE, L, D = 20, 16, 32

class TinyAttnModel(nn.Module):
    def __init__(self, use_pos):
        super().__init__()
        self.tok = nn.Embedding(V_SIZE, D)
        self.pos = nn.Embedding(L, D) if use_pos else None
        self.attn = MultiHeadAttention(D, n_heads=1)
        self.out = nn.Linear(D, V_SIZE)
    def forward(self, seq):                           # seq: [batch, L] token ids
        x = self.tok(seq) + (self.pos(torch.arange(L)) if self.pos is not None else 0)
        z, A = self.attn(x)
        return self.out(z[:, -1]), A                  # predict from the last position only

def batch(b=64):
    s = torch.randint(0, V_SIZE, (b, L))
    return s, s[:, 0]

def train(use_pos, steps=600):
    torch.manual_seed(1)
    m = TinyAttnModel(use_pos); opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    for step in range(steps):
        s, y = batch()
        logits, _ = m(s)
        loss = F.cross_entropy(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
    s, y = batch(500)
    with torch.no_grad():
        acc = (m(s)[0].argmax(-1) == y).float().mean().item()
    return m, acc

m_pos, acc_pos = train(True)
m_nopos, acc_nopos = train(False)
print(f"accuracy with position embeddings   : {acc_pos:.0%}")
print(f"accuracy without position embeddings: {acc_nopos:.0%}   (chance = {1 / V_SIZE:.0%})")

In [ ]:
s, y = batch(1)
with torch.no_grad():
    _, A_trained = m_pos(s)                        # [batch, heads, query, key]
plt.figure(figsize=(7, 1.8))
plt.bar(range(L), A_trained[0, 0, -1]); plt.xticks(range(L))
plt.xlabel("key position"); plt.ylabel("weight"); plt.title("where the last position attends after training")
plt.show()
print(f"weight on position 0: {A_trained[0, 0, -1, 0]:.2f}")

With positions, the last query learns to put almost all its weight on position 0 — one hop, however long the sequence. Without
positions, all copies of a token look identical to the query, so the model cannot tell which one came first; it can only guess the
tokens that occur most often in the sequence, which is far less accurate.

## Exercises
1. Remove the `/ math.sqrt(...)` in `MultiHeadAttention` and retrain with `D = 256`. Does training get slower or less stable?
2. Change the task to *output the token at position 7* or *output the token that appears twice*. Which one needs positions?
3. Add a causal mask to `TinyAttnModel` and change the task to *predict the previous token at every position*. Plot the learned
   attention matrix — you should see a shifted diagonal, a so-called *previous-token head* (look for one in GPT-2 in Notebook 3).

**Back to the [video lecture]({SITE}#p8-s91)**